In [1]:
import pandas as pd
import numpy as np

from scipy.stats import ttest_ind
from scipy.stats import f_oneway
from scipy.stats import pearsonr

In [2]:
master = pd.read_csv("../data/processed/airbnb_featured.csv")

print(master.shape)
master.head()

(96871, 93)


,id,listing_url,scrape_id,last_scraped,source,name,description,neighborhood_overview,picture_url,host_id,...,review_text_count,first_review_date,last_review_date,avg_review_length,neighbourhood_group,neighbourhood_y,revenue_category,price_category,host_experience_years,review_lifetime_days
0,13913,https://www.airbnb.com/rooms/13913,20250914034649,2025-09-16,city scrape,Holiday London DB Room Let-on going,My bright double bedroom with a large window h...,Finsbury Park is a friendly melting pot commun...,https://a0.muscache.com/pictures/miso/Hosting-...,54730,...,55.0,2010-08-18,2025-08-21,234.909091,NaN,Islington,Low,Budget,16.602740,5482.0
1,15400,https://www.airbnb.com/rooms/15400,20250914034649,2025-09-16,city scrape,Bright Chelsea Apartment. Chelsea!,Lots of windows and light. St Luke's Gardens ...,It is Chelsea.,https://a0.muscache.com/pictures/428392/462d26...,60302,...,97.0,2009-12-21,2025-04-05,426.371134,NaN,Kensington and Chelsea,Low,Standard,16.550685,5584.0
2,17402,https://www.airbnb.com/rooms/17402,20250914034649,2025-09-16,city scrape,Very Central Modern 3-Bed/2 Bath By Oxford St W1,"You'll have a great time in this beautiful, cl...","Fitzrovia is a very desirable trendy, arty and...",https://a0.muscache.com/pictures/39d5309d-fba7...,67564,...,56.0,2011-03-21,2024-02-19,379.892857,NaN,Westminster,No Revenue,Premium,16.468493,4718.0
3,24328,https://www.airbnb.com/rooms/24328,20250914034649,2025-09-18,previous scrape,Battersea live/work artist house,"Artist house by SW Battersea Park, bright high...","- Battersea is a quiet family area, easy acces...",https://a0.muscache.com/pictures/9194b40f-c627...,41759,...,95.0,2010-11-15,2025-07-05,395.715789,NaN,Wandsworth,NaN,NaN,16.736986,5346.0
4,36274,https://www.airbnb.com/rooms/36274,20250914034649,2025-09-15,city scrape,Bright 1 bedroom apt off brick lane in Shoreditch,*Update June '25- Pump Installed to improve wa...,NaN,https://a0.muscache.com/pictures/hosting/Hosti...,133271,...,15.0,2011-03-20,2025-09-06,272.266667,NaN,Tower Hamlets,Medium,Standard,16.076712,5284.0


In [3]:
stat_cols = [
    "price",
    "review_scores_rating",
    "number_of_reviews",
    "room_type",
    "host_is_superhost",
    "neighbourhood_cleansed"
]

master[stat_cols].isna().sum()

price                     34908
review_scores_rating      24122
number_of_reviews             0
room_type                     0
host_is_superhost          1766
neighbourhood_cleansed        0
dtype: int64

In [4]:
h1_data = master[
    (master["room_type"].isin(["Entire home/apt", "Private room"])) &
    (master["price"].notna())
]

print(h1_data.shape)

h1_data["room_type"].value_counts()

(61700, 93)


room_type
Entire home/apt    42318
Private room       19382
Name: count, dtype: int64

In [5]:
entire_home_prices = h1_data[
    h1_data["room_type"] == "Entire home/apt"
]["price"]

private_room_prices = h1_data[
    h1_data["room_type"] == "Private room"
]["price"]

t_stat, p_value = ttest_ind(
    entire_home_prices,
    private_room_prices,
    equal_var=False
)

print("T Statistic:", t_stat)
print("P Value:", p_value)

T Statistic: 5.795141308719796
P Value: 6.866494604010075e-09


In [6]:
h2_data = master[
    (master["host_is_superhost"].notna()) &
    (master["review_scores_rating"].notna())
]

print(h2_data.shape)

h2_data["host_is_superhost"].value_counts()

(71209, 93)


host_is_superhost
f    55367
t    15842
Name: count, dtype: int64

In [7]:
superhost_ratings = h2_data[
    h2_data["host_is_superhost"] == "t"
]["review_scores_rating"]

non_superhost_ratings = h2_data[
    h2_data["host_is_superhost"] == "f"
]["review_scores_rating"]

t_stat, p_value = ttest_ind(
    superhost_ratings,
    non_superhost_ratings,
    equal_var=False
)

print("Superhost Mean Rating:", superhost_ratings.mean())
print("Non-Superhost Mean Rating:", non_superhost_ratings.mean())

print("T Statistic:", t_stat)
print("P Value:", p_value)

Superhost Mean Rating: 4.853570256280773
Non-Superhost Mean Rating: 4.637078945942529
T Statistic: 76.03786040912912
P Value: 0.0


In [8]:
h3_data = master[
    master["price"].notna()
].copy()

h3_data["review_group"] = np.where(
    h3_data["number_of_reviews"] > 10,
    "More than 10",
    "10 or Less"
)

print(h3_data["review_group"].value_counts())

review_group
10 or Less      35786
More than 10    26177
Name: count, dtype: int64


In [9]:
high_review_prices = h3_data[
    h3_data["review_group"] == "More than 10"
]["price"]

low_review_prices = h3_data[
    h3_data["review_group"] == "10 or Less"
]["price"]

t_stat, p_value = ttest_ind(
    high_review_prices,
    low_review_prices,
    equal_var=False
)

print("More than 10 Reviews Mean Price:",
      high_review_prices.mean())

print("10 or Less Reviews Mean Price:",
      low_review_prices.mean())

print("T Statistic:", t_stat)
print("P Value:", p_value)

More than 10 Reviews Mean Price: 181.80914543301373
10 or Less Reviews Mean Price: 265.1072486447214
T Statistic: -2.675006809136342
P Value: 0.007475926915068558


In [10]:
top_boroughs = (
    master["neighbourhood_cleansed"]
    .value_counts()
    .head(5)
    .index
)

print(top_boroughs)

Index(['Westminster', 'Tower Hamlets', 'Camden', 'Kensington and Chelsea',
       'Hackney'],
      dtype='str', name='neighbourhood_cleansed')


In [11]:
anova_data = master[
    (master["neighbourhood_cleansed"].isin(top_boroughs)) &
    (master["price"].notna())
]

print(anova_data.shape)

anova_data.groupby(
    "neighbourhood_cleansed"
)["price"].mean()

(24991, 93)


neighbourhood_cleansed
Camden                    216.511547
Hackney                   161.126299
Kensington and Chelsea    336.072148
Tower Hamlets             430.906199
Westminster               342.139405
Name: price, dtype: float64

In [12]:
westminster = anova_data[
    anova_data["neighbourhood_cleansed"] == "Westminster"
]["price"]

tower_hamlets = anova_data[
    anova_data["neighbourhood_cleansed"] == "Tower Hamlets"
]["price"]

camden = anova_data[
    anova_data["neighbourhood_cleansed"] == "Camden"
]["price"]

kensington = anova_data[
    anova_data["neighbourhood_cleansed"] == "Kensington and Chelsea"
]["price"]

hackney = anova_data[
    anova_data["neighbourhood_cleansed"] == "Hackney"
]["price"]

f_stat, p_value = f_oneway(
    westminster,
    tower_hamlets,
    camden,
    kensington,
    hackney
)

print("F Statistic:", f_stat)
print("P Value:", p_value)

F Statistic: 0.9619231113225394
P Value: 0.4270306563296117


In [13]:
calendar = pd.read_csv("../data/raw/calendar.csv")

print(calendar.shape)
calendar.head()

(35357974, 7)


,listing_id,date,available,price,adjusted_price,minimum_nights,maximum_nights
0,1196288722069341420,2025-09-15,f,NaN,NaN,1,1125
1,1196288722069341420,2025-09-16,f,NaN,NaN,1,1125
2,1196288722069341420,2025-09-17,f,NaN,NaN,1,1125
3,1196288722069341420,2025-09-18,f,NaN,NaN,1,1125
4,1196288722069341420,2025-09-19,f,NaN,NaN,1,1125


In [14]:
calendar[["price", "adjusted_price"]].isna().sum()

price             35357974
adjusted_price    35357974
dtype: int64

Confidence Intervals

In [15]:
from scipy.stats import t

price_data = master[
    "price"
].dropna()

n = len(price_data)

mean_price = price_data.mean()

std_price = price_data.std()

margin_error = (
    t.ppf(0.975, n - 1)
    * (std_price / np.sqrt(n))
)

lower = mean_price - margin_error
upper = mean_price + margin_error

print("Sample Size:", n)
print("Mean Price:", mean_price)
print("95% CI Lower:", lower)
print("95% CI Upper:", upper)

Sample Size: 61963
Mean Price: 229.9169827154915
95% CI Lower: 194.97581122776265
95% CI Upper: 264.85815420322035


Correlation Analysis

In [16]:
corr_cols = [
    "price",
    "bedrooms",
    "bathrooms",
    "accommodates",
    "review_scores_rating",
    "occupancy_rate",
    "number_of_reviews",
    "review_text_count",
    "avg_review_length",
    "host_experience_years",
    "estimated_revenue_l365d"
]

corr_df = master[corr_cols]

corr_df.corr(numeric_only=True)

,price,bedrooms,bathrooms,accommodates,review_scores_rating,occupancy_rate,number_of_reviews,review_text_count,avg_review_length,host_experience_years,estimated_revenue_l365d
price,1.000000,0.021513,0.020547,0.018609,-0.012322,-0.010286,-0.006838,-0.017829,0.026577,-0.006988,0.094566
bedrooms,0.021513,1.000000,0.621416,0.741649,0.029882,0.043297,-0.095351,-0.108318,0.093314,0.046240,0.042690
bathrooms,0.020547,0.621416,1.000000,0.497298,0.017883,-0.028790,-0.051236,-0.054799,0.073718,0.022260,0.032434
accommodates,0.018609,0.741649,0.497298,1.000000,-0.036284,-0.110447,-0.045906,-0.068009,0.095617,-0.054545,0.064714
review_scores_rating,-0.012322,0.029882,0.017883,-0.036284,1.000000,0.121363,0.070373,0.070385,-0.165666,0.193701,0.018730
occupancy_rate,-0.010286,0.043297,-0.028790,-0.110447,0.121363,1.000000,-0.049768,-0.077033,0.016312,0.283219,0.001877
number_of_reviews,-0.006838,-0.095351,-0.051236,-0.045906,0.070373,-0.049768,1.000000,1.000000,-0.036274,0.133003,0.084669
review_text_count,-0.017829,-0.108318,-0.054799,-0.068009,0.070385,-0.077033,1.000000,1.000000,-0.036264,0.127864,0.065772
avg_review_length,0.026577,0.093314,0.073718,0.095617,-0.165666,0.016312,-0.036274,-0.036264,1.000000,0.092609,0.006248
host_experience_years,-0.006988,0.046240,0.022260,-0.054545,0.193701,0.283219,0.133003,0.127864,0.092609,1.000000,0.008776


In [17]:
corr_df.corr(
    numeric_only=True
)["price"].sort_values(
    ascending=False
)

price                      1.000000
estimated_revenue_l365d    0.094566
avg_review_length          0.026577
bedrooms                   0.021513
bathrooms                  0.020547
accommodates               0.018609
number_of_reviews         -0.006838
host_experience_years     -0.006988
occupancy_rate            -0.010286
review_scores_rating      -0.012322
review_text_count         -0.017829
Name: price, dtype: float64

## Effect Size Analysis

In [19]:
def cohens_d(group1, group2):
    n1 = len(group1)
    n2 = len(group2)

    s1 = group1.std()
    s2 = group2.std()

    pooled_std = np.sqrt(
        ((n1 - 1) * s1**2 + (n2 - 1) * s2**2) / (n1 + n2 - 2)
    )

    return (group1.mean() - group2.mean()) / pooled_std

In [20]:
h1_cohens_d = cohens_d(
    entire_home_prices,
    private_room_prices
)

print("H1 Cohen's d:", h1_cohens_d)

H1 Cohen's d: 0.03545332939935567


In [21]:
h2_cohens_d = cohens_d(
    superhost_ratings,
    non_superhost_ratings
)

print("H2 Cohen's d:", h2_cohens_d)

H2 Cohen's d: 0.4439740433601297


In [22]:
h3_cohens_d = cohens_d(
    high_review_prices,
    low_review_prices
)

print("H3 Cohen's d:", h3_cohens_d)

H3 Cohen's d: -0.018771681265386468


In [23]:
groups = [
    westminster,
    tower_hamlets,
    camden,
    kensington,
    hackney
]

all_values = pd.concat(groups)

grand_mean = all_values.mean()

ss_between = sum(
    len(group) * (group.mean() - grand_mean) ** 2
    for group in groups
)

ss_total = sum(
    (all_values - grand_mean) ** 2
)

eta_squared = ss_between / ss_total

print("Eta Squared:", eta_squared)

Eta Squared: 0.00015397022393431978


In [24]:
import statsmodels.api as sm

ols_data = master[
    [
        "price",
        "bedrooms",
        "bathrooms",
        "accommodates",
        "occupancy_rate",
        "number_of_reviews",
        "host_experience_years"
    ]
].dropna()

print(ols_data.shape)

X = ols_data.drop(columns=["price"])

y = ols_data["price"]

X = sm.add_constant(X)

ols_model = sm.OLS(y, X).fit()

print("OLS Model Created")

(61748, 7)
OLS Model Created


In [25]:
print(ols_model.summary())

                            OLS Regression Results                            
Dep. Variable:                  price   R-squared:                       0.001
Model:                            OLS   Adj. R-squared:                  0.001
Method:                 Least Squares   F-statistic:                     7.190
Date:                Sat, 20 Jun 2026   Prob (F-statistic):           1.10e-07
Time:                        15:22:02   Log-Likelihood:            -6.0625e+05
No. Observations:               61748   AIC:                         1.213e+06
Df Residuals:                   61741   BIC:                         1.213e+06
Df Model:                           6                                         
Covariance Type:            nonrobust                                         
                            coef    std err          t      P>|t|      [0.025      0.975]
-----------------------------------------------------------------------------------------
const                   129.43